# GroupBy, Aggregation, and Transform

Summarize records at the right level and create group-level metrics safely.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Use `groupby` with `agg` and named aggregations.
- Distinguish row counts from non-missing counts.
- Use `transform` when a group-level statistic must align with original rows.
- Filter groups based on group-level criteria.

In [1]:
import pandas as pd

sales = pd.DataFrame({
    "city": ["Baku", "Baku", "Ganja", "Ganja", "Baku", "Shaki", "Shaki"],
    "category": ["Food", "Books", "Food", "Books", "Food", "Food", "Books"],
    "revenue": [120, 80, 75, 65, 90, 40, 55],
    "units": [6, 2, 5, 3, 4, 2, 1],
})
display(sales)

,city,category,revenue,units
0,Baku,Food,120,6
1,Baku,Books,80,2
2,Ganja,Food,75,5
3,Ganja,Books,65,3
4,Baku,Food,90,4
5,Shaki,Food,40,2
6,Shaki,Books,55,1


## Named aggregations
Named outputs make the resulting DataFrame easier to read and less dependent on multi-level column names.

In [2]:
summary = sales.groupby("city", as_index=False).agg(
    revenue_total=("revenue", "sum"),
    average_order=("revenue", "mean"),
    units_total=("units", "sum"),
    row_count=("revenue", "size"),
)
display(summary.sort_values("revenue_total", ascending=False))

by_city_category = sales.groupby(["city", "category"], as_index=False).agg(
    revenue=("revenue", "sum"), order_count=("revenue", "size")
)
display(by_city_category)

,city,revenue_total,average_order,units_total,row_count
0,Baku,290,96.666667,12,3
1,Ganja,140,70.000000,8,2
2,Shaki,95,47.500000,3,2


,city,category,revenue,order_count
0,Baku,Books,80,1
1,Baku,Food,210,2
2,Ganja,Books,65,1
3,Ganja,Food,75,1
4,Shaki,Books,55,1
5,Shaki,Food,40,1


## `agg` versus `transform`
`agg` reduces each group to a summary. `transform` returns a value for each original row, so it is useful for ratios, deviations from group means, and within-group comparisons.

In [3]:
sales["city_total"] = sales.groupby("city")["revenue"].transform("sum")
sales["share_of_city_revenue"] = sales["revenue"] / sales["city_total"]
sales["above_city_average"] = sales["revenue"] > sales.groupby("city")["revenue"].transform("mean")
display(sales)

# Keep only cities with at least two rows.
city_sizes = sales.groupby("city").size()
large_groups = city_sizes[city_sizes >= 2].index
display(sales[sales["city"].isin(large_groups)])

,city,category,revenue,units,city_total,share_of_city_revenue,above_city_average
0,Baku,Food,120,6,290,0.413793,True
1,Baku,Books,80,2,290,0.275862,False
2,Ganja,Food,75,5,140,0.535714,True
3,Ganja,Books,65,3,140,0.464286,False
4,Baku,Food,90,4,290,0.310345,False
5,Shaki,Food,40,2,95,0.421053,False
6,Shaki,Books,55,1,95,0.578947,True


,city,category,revenue,units,city_total,share_of_city_revenue,above_city_average
0,Baku,Food,120,6,290,0.413793,True
1,Baku,Books,80,2,290,0.275862,False
2,Ganja,Food,75,5,140,0.535714,True
3,Ganja,Books,65,3,140,0.464286,False
4,Baku,Food,90,4,290,0.310345,False
5,Shaki,Food,40,2,95,0.421053,False
6,Shaki,Books,55,1,95,0.578947,True


## Practice
Calculate total revenue, mean revenue, and order count by category. Then add each row's percentage of its category revenue.

In [4]:
category_summary = sales.groupby("category", as_index=False).agg(
    total_revenue=("revenue", "sum"), mean_revenue=("revenue", "mean"), orders=("revenue", "size")
)
sales["category_share"] = sales["revenue"] / sales.groupby("category")["revenue"].transform("sum")
display(category_summary)
display(sales[["category", "revenue", "category_share"]])

,category,total_revenue,mean_revenue,orders
0,Books,200,66.666667,3
1,Food,325,81.250000,4


,category,revenue,category_share
0,Food,120,0.369231
1,Books,80,0.400000
2,Food,75,0.230769
3,Books,65,0.325000
4,Food,90,0.276923
5,Food,40,0.123077
6,Books,55,0.275000
